# VL-JEPA: Real Models (No device_map)

**Fixed for accelerate issues** - manually moves models to device

In [ ]:
%pip install torch torchvision torchaudio transformers datasets huggingface-hub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, AutoVideoProcessor, LlamaForCausalLM
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 1. Load V-JEPA2 Vision Encoder

In [ ]:
print('Loading V-JEPA2 vision encoder...')

# Load V-JEPA2 model and processor
model_id = 'facebook/vjepa2-vitl-fpc64-256'
print(f'Loading {model_id}...')

vision_processor = AutoVideoProcessor.from_pretrained(model_id)
vision_model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to(device)

print(f'✓ Loaded V-JEPA2: {model_id}')

# Freeze
for param in vision_model.parameters():
    param.requires_grad = False
vision_model.eval()

vision_dim = vision_model.config.hidden_size
print(f'Hidden dim: {vision_dim}')
print(f'Params: {sum(p.numel() for p in vision_model.parameters())/1e6:.1f}M (frozen)')

## 2. Load Llama Predictor

In [ ]:
print('\nLoading Llama-3.2-1B...')
llama = LlamaForCausalLM.from_pretrained(
    'meta-llama/Llama-3.2-1B',
    torch_dtype=torch.float16
).to(device)

# Extract last 8 layers
predictor_layers = nn.ModuleList(llama.model.layers[8:16])
predictor_dim = llama.config.hidden_size

print(f'✓ Extracted layers 8-15')
print(f'Hidden dim: {predictor_dim}')
print(f'Params: {sum(p.numel() for p in predictor_layers.parameters())/1e6:.1f}M')

# Input projection
predictor_proj_in = nn.Linear(vision_dim, predictor_dim).to(device)
print(f'Projection: {vision_dim} → {predictor_dim}')

## 3. Load Gemma Text Encoder

In [ ]:
print('\nLoading Gemma-2-2B...')
text_model = AutoModel.from_pretrained(
    'google/gemma-2-2b',
    torch_dtype=torch.float16
).to(device)
text_tokenizer = AutoTokenizer.from_pretrained('google/gemma-2-2b')

if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
    print('Set pad_token = eos_token')

text_dim = text_model.config.hidden_size
print(f'✓ Loaded Gemma')
print(f'Hidden dim: {text_dim}')
print(f'Params: {sum(p.numel() for p in text_model.parameters())/1e6:.1f}M')

## 4. Load MSR-VTT Dataset

## 5. Dataset Class

In [ ]:
print('\nLoading dataset...')
try:
    dataset = load_dataset('AlexZigma/msr-vtt', split='train')
    print(f'✓ MSR-VTT: {len(dataset)} videos')
except:
    print('MSR-VTT unavailable, using COCO...')
    dataset = load_dataset('HuggingFaceM4/COCO', split='train')
    print(f'✓ COCO: {len(dataset)} images')

sample = dataset[0]
print(f'\nKeys: {list(sample.keys())}')
if 'caption' in sample:
    print(f'Caption: {sample["caption"][:80]}...')
elif 'sentences' in sample:
    print(f'Caption: {sample["sentences"]["raw"][0][:80]}...')

In [ ]:
class VideoTextDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, processor, num_frames=16):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.processor = processor
        self.num_frames = num_frames
        self.has_image = 'image' in hf_dataset[0]
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Load image as repeated frames
        if self.has_image:
            img = sample['image'].convert('RGB')
            frames = [img] * self.num_frames
        else:
            frames = [Image.new('RGB', (224, 224))] * self.num_frames
        
        # Process for vision model (video processor)
        pixels = self.processor(videos=frames, return_tensors='pt')['pixel_values_videos']
        
        # DEBUG: Show processor output shape
        if idx == 0:  # Only print for first sample
            print(f'[DEBUG] Processor output shape: {pixels.shape}')
        
        # Remove all batch dimensions - processor may return [1, 1, T, C, H, W] or [1, T, C, H, W]
        while pixels.dim() > 4:  # Target: [T, C, H, W]
            if pixels.shape[0] == 1:
                pixels = pixels.squeeze(0)
            else:
                break
        
        # DEBUG: Show shape after squeeze
        if idx == 0:
            print(f'[DEBUG] After squeeze shape: {pixels.shape}')
        
        # Get caption
        if 'caption' in sample:
            caption = sample['caption']
        elif 'sentences' in sample:
            caption = sample['sentences']['raw'][0]
        else:
            caption = 'A video.'
        
        # Tokenize
        tokens = self.tokenizer(
            caption,
            return_tensors='pt',
            padding='max_length',
            max_length=77,
            truncation=True
        )
        
        return {
            'pixels': pixels,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }

train_dataset = VideoTextDataset(dataset, text_tokenizer, vision_processor)
print(f'\n✓ Dataset ready: {len(train_dataset)} samples')

## 6. VL-JEPA Model

In [ ]:
class VLJEPA(nn.Module):
    def __init__(self, vision, pred_layers, proj_in, text, embed_dim=2048):
        super().__init__()
        self.vision = vision
        self.pred_layers = pred_layers
        self.proj_in = proj_in
        self.text = text
        
        self.pred_proj = nn.Linear(predictor_dim, embed_dim).to(device)
        self.text_proj = nn.Linear(text_dim, embed_dim).to(device)
        # Initialize temperature as float16
        self.temperature = nn.Parameter(torch.tensor(0.07, dtype=torch.float16).to(device))
    
    def forward(self, pixels, input_ids, attention_mask):
        print(f"[MODEL] Input pixels.shape: {pixels.shape}")
        # Input pixels: [B, T, C, H, W]
        # V-JEPA2 expects: [B, C, T, H, W]
        pixels = pixels.permute(0, 2, 1, 3, 4).contiguous()  # [B, T, C, H, W] -> [B, C, T, H, W]
        print(f"[MODEL] After permute: {pixels.shape}")
        B, C, T, H, W = pixels.shape
        
        # Encode video (frozen) - V-JEPA2 expects [B, C, T, H, W]
        with torch.no_grad():
            pixels = pixels.to(dtype=torch.float16)
            print(f"[MODEL] After .to(): {pixels.shape}")
            v_out = self.vision(pixel_values_videos=pixels)
            # V-JEPA2 output: [B, hidden_dim] (already pooled) or [B, T, hidden_dim]
            if hasattr(v_out, 'pooler_output') and v_out.pooler_output is not None:
                visual = v_out.pooler_output  # [B, hidden_dim]
                # Expand to [B, T, hidden_dim] for predictor
                visual = visual.unsqueeze(1).expand(-1, T, -1)
            else:
                visual = v_out.last_hidden_state  # Should be [B, T, hidden_dim]
                if visual.dim() == 2:  # If [B, hidden_dim]
                    visual = visual.unsqueeze(1).expand(-1, T, -1)
        
        # Predictor: [B, T, hidden_dim]
        h = self.proj_in(visual)
        for layer in self.pred_layers:
            h = layer(h, use_cache=False)[0]
        pred = F.normalize(self.pred_proj(h), dim=-1)
        
        # Text encoder
        text_out = self.text(input_ids=input_ids, attention_mask=attention_mask)
        target = F.normalize(self.text_proj(text_out.last_hidden_state), dim=-1)
        
        return pred, target
    
    def compute_loss(self, pred, target, mask):
        # Pool
        pred_pool = pred.mean(dim=1)
        target_pool = (target * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        pred_pool = F.normalize(pred_pool, dim=-1)
        target_pool = F.normalize(target_pool, dim=-1)
        
        # InfoNCE
        sim = torch.matmul(pred_pool, target_pool.T) / self.temperature
        labels = torch.arange(sim.shape[0], device=sim.device)
        loss_v2t = F.cross_entropy(sim, labels)
        loss_t2v = F.cross_entropy(sim.T, labels)
        return (loss_v2t + loss_t2v) / 2

model = VLJEPA(vision_model, predictor_layers, predictor_proj_in, text_model)

# Convert all trainable parameters to float16
model.pred_proj = model.pred_proj.half()
model.text_proj = model.text_proj.half()
model.proj_in = model.proj_in.half()
for layer in model.pred_layers:
    layer.half()
model.text.half()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M | Frozen: {(total-trainable)/1e6:.1f}M')
print(f'\nDtype check:')
print(f'  Vision: {next(model.vision.parameters()).dtype}')
print(f'  Predictor: {next(model.pred_layers.parameters()).dtype}')
print(f'  Text: {next(model.text.parameters()).dtype}')
print(f'  Temperature: {model.temperature.dtype}')

## 7. Training Setup

In [ ]:
# DataLoader
batch_size = 4
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)
print(f'DataLoader: {len(train_loader)} batches\n')

# Optimizer (differential learning rates)
base_lr = 1e-4
text_lr = base_lr * 0.05

text_params = list(model.text.parameters()) + list(model.text_proj.parameters())
other_params = [p for p in model.parameters() if p.requires_grad and not any(p is tp for tp in text_params)]

optimizer = torch.optim.AdamW([
    {'params': other_params, 'lr': base_lr},
    {'params': text_params, 'lr': text_lr}
], weight_decay=0.01)

print(f'Base LR: {base_lr}')
print(f'Text LR: {text_lr}')

## 8. Training Loop

In [ ]:
max_steps = 500
log_interval = 50

print(f'\nTraining for {max_steps} steps...\n')

model.train()
global_step = 0

for batch in tqdm(train_loader, total=max_steps):
    if global_step >= max_steps:
        break
    
    pixels = batch['pixels'].to(device)
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    
    # Forward
    pred, target = model(pixels, input_ids, attention_mask)
    loss = model.compute_loss(pred, target, attention_mask)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    global_step += 1
    
    if global_step % log_interval == 0:
        print(f'Step {global_step} | Loss: {loss.item():.4f} | Temp: {model.temperature.item():.4f}')

print('\n✓ Training complete!')

## 9. Save Checkpoint

In [ ]:
checkpoint = {
    'predictor_layers': predictor_layers.state_dict(),
    'proj_in': predictor_proj_in.state_dict(),
    'pred_proj': model.pred_proj.state_dict(),
    'text_model': text_model.state_dict(),
    'text_proj': model.text_proj.state_dict(),
    'temperature': model.temperature.data,
    'optimizer': optimizer.state_dict(),
}

torch.save(checkpoint, 'vljepa_checkpoint.pt')
print('✓ Saved to vljepa_checkpoint.pt')